# Raman spectrum analysis: a first measurement

Run the cells in order. This notebook loads a bundled CZTS measurement, explains the processing choices, plots the fit and exports the results. It fits peaks; it does **not** identify compounds.

Locally, install the project and open this notebook from the repository. In Google Colab, the setup below installs from the default GitHub branch. The Colab path is available after the accompanying PR is merged. Uploaded measurements are processed on Google's infrastructure; use a local notebook for data you cannot upload.

In [ ]:
from pathlib import Path
import subprocess
import sys

candidates = [Path.cwd(), Path.cwd().parent]
repo = next((p for p in candidates if (p / "pyproject.toml").is_file()
             and (p / "src/raman_spectroscopy").is_dir()), None)
if repo is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/btjones-me/raman_spectroscopy.git@master"])
else:
    # Use the checkout under review rather than installing GitHub's older code.
    sys.path.insert(0, str(repo / "src"))

import numpy as np
from matplotlib import pyplot as plt
from raman_spectroscopy import AnalysisSettings, analyse_spectrum, load_spectrum, save_result
from raman_spectroscopy.plotting import plot_result

## Load one example

Inputs have exactly two numeric columns: Raman shift (cm⁻¹) and intensity. This file has no header. CSV files default to commas; use `skiprows=1` for a one-line header. The bundled shifts descend, which is supported.

In [ ]:
relative = Path("Raman Spectroscopy/CZTS_data/CZTS_111116/B21/B21_1.txt")
if repo is not None:
    sample = repo / relative
else:
    from urllib.request import urlretrieve
    sample = Path("B21_1.txt")
    if not sample.exists():
        urlretrieve("https://raw.githubusercontent.com/btjones-me/raman_spectroscopy/"
                    "f9197e24db4a87eb179307cc025d9b6c9d372b22/"
                    "Raman%20Spectroscopy/CZTS_data/CZTS_111116/B21/B21_1.txt", sample)
x, intensity = load_spectrum(sample)
print(f"{sample.name}: {len(x)} points, {x.min():.2f}–{x.max():.2f} cm⁻¹")

## Choose settings and fit

The default `legacy` baseline reproduces the dissertation's quadratic addition and degree-2 baseline subtraction. Corrected intensity is normalised to a maximum of 9.5 a.u. Five is a requested maximum number of detected peaks, not a confirmed physical component count.

Try `baseline="polynomial"` to apply PeakUtils directly or `baseline="none"` to skip baseline subtraction. These change the method and need evaluation for your experiment. The fit remains unconstrained.

In [ ]:
settings = AnalysisSettings(n_peaks=5, baseline="legacy")
result = analyse_spectrum(x, intensity, settings)
print(f"RMSE: {result.rmse:.4f} a.u.")
print("Warnings:", result.warnings or "None reported; inspect the fit regardless.")
print("centre (cm^-1), height (a.u.), HWHM (cm^-1)")
print(np.round(result.parameters, 4))

## Inspect the result

Check baseline shape, individual fitted components and residual structure across the **full measured range**. HWHM is half the full width at half maximum; exported FWHM is twice HWHM. Positive reported widths preserve the original unconstrained model, which squares width. A small residual alone does not prove a unique or physically correct decomposition.

In [ ]:
figure = plot_result(result, title=sample.name)
plt.show()
plt.close(figure)

## Save reproducible outputs

A new folder is created for every run. It contains peak and spectrum CSVs, a diagnostic figure, and JSON with settings, input checksum, covariance and software versions. Standard errors are approximate local-fit estimates, not calibrated experimental uncertainty.

In [ ]:
from datetime import datetime, timezone
output = Path("results") / ("notebook-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
save_result(result, output, source=sample)
figure = plot_result(result, title=sample.name)
figure.savefig(output / "fit.png", dpi=150)
plt.close(figure)
print(f"Saved to {output.resolve()}")

## Optional: use your own spectrum

Set `use_own_data = True` to select an uploaded file in Colab. Locally, set `own_path` to your measurement instead. Set `header_rows` to the number of header lines. The code does not overwrite uploaded filenames: Colab upload bytes go into a new temporary folder.

In [ ]:
use_own_data = False
own_path = None  # Local example: Path("/path/to/measurement.csv")
header_rows = 0
if use_own_data:
    if "google.colab" in sys.modules:
        from google.colab import files
        from tempfile import mkdtemp
        import os
        upload_directory = Path(mkdtemp(prefix="raman-upload-"))
        previous = Path.cwd()
        try:
            os.chdir(upload_directory)
            uploaded = files.upload()
        finally:
            os.chdir(previous)
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one spectrum for this example")
        own_path = upload_directory / next(iter(uploaded))
    if own_path is None:
        raise ValueError("Set own_path to your measurement")
    own_x, own_y = load_spectrum(own_path, skiprows=header_rows)
    own_result = analyse_spectrum(own_x, own_y, settings)
    own_output = output / "own-spectrum"
    save_result(own_result, own_output, source=own_path, input_options={"skiprows": header_rows})
    figure = plot_result(own_result, title=Path(own_path).name)
    figure.savefig(own_output / "fit.png", dpi=150)
    plt.show()
    plt.close(figure)
    print("Warnings:", own_result.warnings)

## Optional: download results from Colab

Set `download_results = True` to download all outputs from this run as a ZIP. For batch analysis and repeat averaging, see the repository README.

In [ ]:
download_results = False
if download_results:
    import shutil
    archive = shutil.make_archive(str(output), "zip", output)
    if "google.colab" in sys.modules:
        from google.colab import files
        files.download(archive)
    else:
        print(f"Archive: {archive}")